# Thompson Sampling pour les bandits manchots non-stationnaires

**Auteur:** François Cinotti

---

Le **Thompson Sampling** est une méthode puissante de résolution de « bandits manchots », largement utilisée pour la publicité en ligne et dans les systèmes de recommandation. Sa force réside dans sa capacité à équilibrer efficacement l'exploration et l'exploitation : il commence par des choix très exploratoires qui lui permettent d'apprendre, avant de converger vers une exploitation quasi exclusive de la meilleure option. Cependant, face à des bandits non-stationnaires, le Thompson Sampling standard est peu performant car il ne peut pas facilement revenir à une phase d'exploration. Dans ce tutoriel, je vous présenterai deux méthodes permettant de pallier cette faiblesse : le **Dynamic Thompson Sampling** (Gupta et al. 2011) et le **Sliding-Window Thompson Sampling** (Trovo et al. 2020), que j'ai utilisées pour comparaison avec le comportement animal dans un article portant sur le méta-apprentissage animal (Cinotti et al. 2024).

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

# Fixed random seed for reproducibility, remove to see different outcomes
np.random.seed(42)

# Use a clean plot style
plt.style.use('seaborn-v0_8-whitegrid')

---
## La distribution Beta

### Propriétés de base
Supposons que je vous donne une pièce et que vous souhaitiez vérifier qu'elle est équilibrée. Vous la lanceriez probablement plusieurs fois, compteriez le nombre de faces et de piles obtenus, et vous feriez votre avis selon que les deux nombres sont à peu près égaux ou non. Mais combien de fois faut-il la lancer, et que signifie « à peu près égaux » ? La **distribution beta** peut vous aider à répondre à ces questions de manière rigoureuse, si vous êtes un mathématicien endurci, ou mieux encore, visuellement, si vous êtes un simple mortel. La distribution beta est une fonction de densité de probabilité définie pour des valeurs comprises entre 0 et 1, qui prend généralement la forme d'une cloche plus ou moins large et asymétrique. Ce n'est pas l'endroit pour entrer dans sa formule et ses propriétés. Pour nos besoins, tout ce qu'il faut savoir, c'est qu'elle possède deux **paramètres de forme** appelés **$\alpha$**, qui tire le pic de la fonction vers 1, et **$\beta$** (un choix de nom un peu déroutant) qui le tire vers 0. En effet, la position du pic, ou mode de la distribution, que l'on peut approximativement comprendre comme la valeur la plus probable lors d'un échantillonnage, vaut ${\alpha -1} \over {\alpha + \beta - 2}$. Par ailleurs, la somme de $\alpha$ et $\beta$ resserre le pic. Les probabilités étant elles-mêmes comprises entre 0 et 1, la distribution beta se prête naturellement à la représentation de croyances sur une probabilité, une probabilité d'une probabilité en quelque sorte. Sur le plan mathématique, « se prête naturellement » est un euphémisme : la distribution beta est en réalité le *conjugate prior* des probabilités binomiales (probabilités pour des résultats binaires comme les lancers de pièce), et possède des propriétés dépassant le cadre de ce notebook qui la rendent particulièrement idéale pour représenter des croyances sur une probabilité binomiale. Fort heureusement pour nous, cette fonction et différentes méthodes qui y sont liées sont déjà intégrées dans la bibliothèque Python *scipy.stats* (voir l'import ci-dessus).

### Croyances a priori
Pour tirer le meilleur parti des distributions beta dans notre exemple, il faut d'abord choisir une croyance **a priori** ou *prior* en anglais: une distribution beta initiale de paramètres $\alpha_0$ et $\beta_0$. Un bon choix, qui refléterait votre confiance en moi, serait une distribution déjà centrée sur 0,5, par exemple $\alpha_0 = \beta_0 = 5$. Ce serait un prior assez informatif car la forme de la fonction est déjà une cloche bien définie. D'un autre côté, je ne suis qu'un inconnu sur internet, et vous êtes donc légitimement méfiant : vous vous attendez à ce que la probabilité de tomber sur face soit très proche de 0 ou de 1, ce que vous pourriez exprimer en posant $\alpha_0=\beta_0=1/2$ qui produit une distribution en U (ce prior est connu sous le nom de *prior de Jeffreys*, et a la propriété remarquable d'être invariant par reparamétrisation : si, au lieu de la probabilité de tomber sur face, vous raisonniez en termes de rapports de cotes, changer de paramètre n'aurait pas d'impact). Une troisième possibilité, que nous utiliserons par la suite, est un prior agnostique et plat, ou **prior uniforme** : la probabilité de tomber sur face peut prendre n'importe quelle valeur entre 0 et 1 ; cela s'obtient en posant $\alpha_0=\beta_0=1$.

In [ ]:
# Plot different prior beliefs

x = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('A selection of prior beliefs', fontsize=13)

# informative prior
axes[0].plot(x, beta_dist.pdf(x, 5, 5), 'k', lw = 2)
axes[0].set_ylabel('Density')
axes[0].set_xlabel('Probability of heads')
axes[0].set_title('Trusting prior')

# Jeffreys prior
axes[1].plot(x, beta_dist.pdf(x, 0.5, 0.5), 'k', lw = 2)
axes[1].set_xlabel('Probability of heads')
axes[1].set_title('Suspicious prior')

# uniform prior
axes[2].plot(x, beta_dist.pdf(x, 1, 1), 'k', lw=2)
axes[2].set_xlabel('Probability of heads')
axes[2].set_title('Agnostic prior')
axes[2].set_ylim([0,2])

plt.tight_layout()
plt.show()

### Mise à jour de vos croyances
Votre choix de prior est fait; imaginons maintenant que vous ayez réalisé deux expériences différentes : dans la première, vous avez lancé la pièce 10 fois et obtenu 6 faces et 4 piles, ce qui paraît raisonnable pour une pièce équilibrée, et dans la seconde, vous l'avez lancée 100 fois et observé 60 faces contre 40 piles. Notez que le rapport faces/piles est identique dans les deux expériences; ce qui change, c'est le nombre total de lancers effectués. Pour représenter votre croyance sur la probabilité de tomber sur face après ces deux expériences, il vous suffit d'injecter le nombre de faces dans la valeur de $\alpha$ et le nombre de piles dans $\beta$:

- $\alpha = \alpha_0 + n_{faces}$
- $\beta = \beta_0 + n_{piles}$

In [ ]:
# Plot posterior beliefs after experiment 1 for different prior beliefs
x = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Posterior beliefs', fontsize=13)

# informative prior
axes[0].plot(x, beta_dist.pdf(x, 5+6, 5+4), 'steelblue', lw = 2, label = 'after 10 flips')
axes[0].plot(x, beta_dist.pdf(x, 5+60, 5+40), 'tomato', lw = 2, label = 'after 100 flips')
axes[0].set_ylabel('Density')
axes[0].set_xlabel('Probability of heads')
axes[0].set_title('Trusting prior')
axes[0].legend()

# Jeffreys prior
axes[1].plot(x, beta_dist.pdf(x, 0.5+6, 0.5+4), 'steelblue', lw = 2, label = 'after 10 flips')
axes[1].plot(x, beta_dist.pdf(x, 0.5+60, 0.5+40), 'tomato', lw = 2, label = 'after 100 flips')
axes[1].set_xlabel('Probability of heads')
axes[1].set_title('Suspicious prior')
axes[1].legend()

# uniform prior
axes[2].plot(x, beta_dist.pdf(x, 1+6, 1+4), 'steelblue', lw=2, label = 'after 10 flips')
axes[2].plot(x, beta_dist.pdf(x, 1+60, 1+40), 'tomato', lw=2, label = 'after 100 flips')
axes[2].set_xlabel('Probability of heads')
axes[2].set_title('Agnostic prior')
axes[2].legend()

plt.tight_layout()
plt.show()

Plusieurs leçons méritent d'être tirées ici. Premièrement, le nombre d'observations est vraiment déterminant. Pas tant pour la position des pics — qui, à l'exception du prior informatif, n'est pas visiblement différente entre les deux expériences (rappelons que le mode est égal à ${\alpha -1} \over {\alpha + \beta - 2}$, ce qui serait exactement le même dans les deux expériences n'était-ce les petits paramètres initiaux) — mais bien pour l'étroitesse ou la variance de la distribution. Si vous vous étiez arrêté à 10 lancers, une vraie probabilité de faces de 0,5 serait tout à fait crédible, mais aussi une large gamme d'autres valeurs, entre 0,3 et 0,8... En revanche, après 100 essais, malgré le même ratio de faces, 0,5 est à la limite des valeurs probables, qui se situent entre 0,5 et 0,7. Deuxièmement, contrairement à ce que les critiques de la **statistique bayésienne** pourraient avancer (vous avez, incidemment, suivi une introduction aux statistiques bayésiennes pour débutants), le choix du prior perd rapidement tout impact significatif sur le résultat final. Même avec seulement 10 observations, les trois fonctions de densité se chevauchent considérablement pour les trois priors différents, et encore davantage après 100 observations.

---
## Les bandits manchots

Oubliez maintenant les lancers de pièce ; place aux choses sérieuses ! Imaginez que vous vous trouviez dans un casino face à trois machines à sous (les « bras »). Chaque machine délivre une récompense avec une probabilité inconnue. Votre objectif est naturellement de maximiser votre gain total sur une série d'essais.
C'est le classique **dilemme exploration-exploitation** :
- **Exploiter**: c'est choisir le bras que vous croyez actuellement être le meilleur
- **Explorer**: c'est essayer d'autres bras au cas où l'un d'eux serait en réalité meilleur.

Une stratégie gourmande (toujours exploiter) risque de vous bloquer sur un bras sous-optimal, tandis qu'une stratégie purement aléatoire (toujours explorer) gaspille des essais sur de mauvais bras. Ce qu'il vous faut, c'est une stratégie qui équilibre judicieusement ce **compromis exploration-exploitation** : explorer quand l'incertitude est grande, exploiter quand vous savez quel est le meilleur bras. C'est précisément la tâche pour laquelle le **Thompson Sampling** a été conçu.
Dans les sections suivantes, nous travaillerons avec un exemple de bandit à trois bras dont les probabilités de récompense sont les suivantes :

| Bras | Probabilité de récompense |
|-----|-------------------|
| 1   | 0.6 ✓ (le meilleur)      |
| 2   | 0.2               |
| 3   | 0.2               |

A chaque essai, tirer sur le bras $i$ donne une récompense de 1 avec une probabilité $p_i$, et 0 sinon.



In [ ]:
# Define the true reward probabilities for each arm
# The agent does NOT have access to these — they are the ground truth we use to simulate the task
TRUE_PROBS = np.array([0.6, 0.2, 0.2])
n_arms = len(TRUE_PROBS)

def pull_arm(arm, probs):
    """
    Simulate pulling an arm.
    Returns 1 (reward) with probability probs[arm], else 0.
    """
    return int(np.random.rand() < probs[arm])

---
## Le Thompson Sampling standard

### L'idée maîtresse

Au lieu de stocker une estimation unique de la probabilité de récompense de chaque bras, le Thompson Sampling les représente à l'aide de distributions beta qui donnent une image complète de toutes les valeurs possibles de cette probabilité. Pour exploiter pleinement cette riche information, le choix du bras à chaque essai repose sur des **échantillons tirés aléatoirement de chaque distribution**, et le bras ayant produit le plus grand échantillon est sélectionné. Au début, lorsque les distributions a priori sont encore très larges (typiquement un prior uniforme, la convention que j'adopterai pour la suite de ce notebook), n'importe quel bras peut produire un grand échantillon, ce qui favorise l'exploration. À mesure que les observations s'accumulent, les distributions beta se resserrent et s'écartent les unes des autres, de sorte que l'exploitation du meilleur bras augmente progressivement.

### L'algorithme

Plus précisément, chaque essai se décompose en 4 étapes:

1. **Echantilloner** chaque bras
2. **Sélectionner** le bras i avec l'échantillon le plus grand
3. **Observer** la récompense obtenue (0 ou 1)
4. **Mettre à jour** la distribution beta du bras sélectionné:
   - Si récompense = 1: $\alpha_i \leftarrow \alpha_i + 1$
   - Si récompense = 0: $\beta_i \leftarrow \beta_i + 1$

Appliquons maintenant le Thompson Sampling à notre bandit à trois bras et voyons dans quelle mesure il identifie correctement chaque bras:

In [ ]:
def thompson_sampling(n_trials, prob_schedule):
    """
    Run vanilla Thompson Sampling on a bandit task.
    
    Parameters
    ----------
    n_trials : Number of trials to run
    prob_schedule : array of shape (n_trials, n_arms)
        True reward probabilities at each trial
    
    Returns
    -------
    choices : Which arm was chosen on each trial
    rewards : Reward received on each trial
    alphas : Evolution of alpha parameters over trials
    betas : Evolution of beta parameters over trials
    """
    
    # initialise beta distribution parameters
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    
    # trial-by-trial storage of results
    choices = np.zeros(n_trials, dtype=int)
    rewards = np.zeros(n_trials, dtype=int)
    alphas = np.zeros((n_trials, n_arms))
    betas = np.zeros((n_trials, n_arms))
    
    for t in range(n_trials):
        
        # step 1: sample one value from each arm's beta distribution
        samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
        
        # step 2: choose the arm with the highest sample
        choice = np.argmax(samples)
        
        # step 3: pull the arm and observe the reward
        reward = pull_arm(choice, prob_schedule[t]) # output = 1 or 0
        
        # step 4: update the Beta distribution of the chosen arm
        alpha[choice] += reward
        beta[choice] += 1 - reward
        
        # store results of this trial
        choices[t] = choice
        rewards[t] = reward
        alphas[t] = alpha
        betas[t] = beta
    
    return choices, rewards, alphas, betas

# Run Thompson Sampling for 500 trials
N_TRIALS = 500
prob_schedule = np.tile(TRUE_PROBS, (N_TRIALS, 1))
choices, rewards, alphas, betas = thompson_sampling(N_TRIALS, prob_schedule)

print(f"Final arm selection counts: Arm 1: {np.sum(choices==0)}, "
      f"Arm 2: {np.sum(choices==1)}, Arm 3: {np.sum(choices==2)}")

### Resultats

Le décompte final des bras nous indique que le bras 1 a effectivement été sélectionné la plupart du temps, avec quelques phases d'exploration des deux autres bras. Nous allons maintenant tracer l'évolution temporelle de cette expérience (les deux cellules sont intentionnellement séparées pour vous permettre de modifier les paramètres graphiques sans relancer la simulation):

In [ ]:
# ---------------- User parameters to play with -----------------
window = 20 # size of smoothing window for performance
snapshot_T = 500 # trial at which to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['steelblue', 'tomato', 'green']

# ---------------------------------------------------------------
# LEFT FIGURE: PERFORMANCE + RASTER PLOT BELOW

# Compute the proportion of trials on which the best arm was chosen

best_arm_chosen = (choices == 0).astype(float)
performance = np.convolve(best_arm_chosen, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes[0].plot(performance, color='steelblue', lw=2)
axes[0].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes[0].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes[0].set_title('Performance')
axes[0].legend()
axes[0].set_ylim(-bottom_margin, 1.05)
axes[0].set_xlim(0, len(choices))

# horizontal separator between curve area and raster
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes[0].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices == arm_idx)[0]
    axes[0].vlines(
            trial_indices,
            y_center - tick_height * 0.5,
            y_center + tick_height * 0.5,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes[0].text(
            0 - 25, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# ---------------------------------------------------------------
# RIGHTT FIGURE: SNAPSHOT OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)

for i in range(n_arms):
    a, b = alphas[snapshot_T-1, i], betas[snapshot_T-1, i]
    axes[1].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={TRUE_PROBS[i]})')
    axes[1].axvline(TRUE_PROBS[i], color=colors[i], linestyle='--', alpha=0.5)
axes[1].set_xlabel('Reward probability')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Beta distributions after {snapshot_T} trials')
axes[1].legend()

plt.tight_layout()
plt.show()

### Interprétation

Note : si vous relancez ce notebook avec une graine aléatoire différente, les détails de l'interprétation suivante peuvent varier.
Après environ 100 essais d'exploration, l'agent choisit presque exclusivement le bras 1. Remarquez comment, à la fin de l'expérience, au bout de 500 essais, la distribution beta du bras 1 est assez étroite et proche de la vraie valeur de 0,6, tandis qu'il subsiste encore beaucoup d'incertitude sur les bras 2 et 3, surtout le 3. En effet, après une phase initiale d'exploration, le chevauchement entre les trois distributions est devenu si faible que sélectionner l'un de ces deux autres bras était extrêmement peu probable ; leurs distributions restent donc assez incertaines. À ce stade, réduire davantage cette incertitude n'a plus vraiment d'importance : tout ce qui compte pour l'agent, c'est que ces bras sont définitivement moins bons que le bras 1. En revanche, parce que l'algorithme continue de choisir le bras 1 sans jamais cesser d'apprendre, la distribution beta de ce dernier est devenue très étroite à mesure que les preuves s'accumulent — avec des conséquences que nous allons explorer dans la section suivante.

---
## Les bandits manchots non-stationnaires

### Le problème

Dans le monde réel — et dans de nombreuses expériences de neurosciences — les probabilités de récompense changent au fil du temps. Dans l'expérience sur les rats de Cinotti et al. (2024), par exemple, l'identité du meilleur levier changeait toutes les 24 tentatives sans avertissement.
Le problème fondamental du Thompson Sampling standard est qu'il accumule les preuves indéfiniment. Comme nous venons de le voir, après de nombreux essais, les distributions beta deviennent très étroites; lorsque les probabilités de récompense changent, l'agent est lent à s'en apercevoir car il faut beaucoup de nouvelles observations pour faire bouger une distribution déjà très concentrée. De plus, même si la distribution se décale progressivement, elle ne s'élargit jamais, et à long terme la distribution représente une moyenne sur l'ensemble du passé, si bien que, si — comme dans l'exemple suivant — le bras correct alterne entre les différentes options, elles convergent toutes vers la même distribution, empêchant toute discrimination utile du meilleur bras actuel.

### Tâche non-stationnaire
Pour illustrer ce problème, nous allons simuler une tâche où les probabilités de récompense alternent toutes les 500 tentatives et y appliquer le Thompson Sampling :

| Bloc | Bras 1 | Bras 2 | Bras 3 |
|-------|-------|-------|-------|
| 1     | **0.6** | 0.2 | 0.2 |
| 2     | 0.2 | **0.6** | 0.2 |
| 3     | 0.2 | 0.2 | **0.6** |
| 4     | **0.6** | 0.2 | 0.2 |
| ...   | ... | ... | ... |

In [ ]:
def make_nonstationary_schedule(n_trials, block_size=500):
    """
    Generate a schedule of reward probabilities that rotate every block_size trials.
    The best arm rotates: 1 -> 2 -> 3 -> 1 -> ...
    
    Returns
    -------
    prob_schedule : array of shape (n_trials, n_arms)
        True reward probabilities at each trial
    best_arm_schedule : array of int
        Which arm is best at each trial
    """
    prob_schedule = np.zeros((n_trials, n_arms))
    best_arm_schedule = np.zeros(n_trials, dtype=int)
    
    for t in range(n_trials):
        # Which block are we in?
        block = (t // block_size) % n_arms
        
        # The best arm rotates with each block
        probs = [0.2, 0.2, 0.2]
        probs[block] = 0.6
        
        prob_schedule[t] = probs
        best_arm_schedule[t] = block
    
    return prob_schedule, best_arm_schedule

# Generate the schedule
N_TRIALS_NS = 3000  # 6 blocks of 100 trials
BLOCK_SIZE = 500
prob_schedule, best_arm_schedule = make_nonstationary_schedule(N_TRIALS_NS, BLOCK_SIZE)

print("Reward probabilities in each block:")
for b in range(6):
    t = b * BLOCK_SIZE
    print(f"  Block {b+1} (trials {t+1}-{t+BLOCK_SIZE}): → best arm: {best_arm_schedule[t]+1}")
print("=" * 80)

# Run vanilla Thompson Sampling on the non-stationary task
choices, rewards, alphas, betas = thompson_sampling(N_TRIALS_NS, prob_schedule)

print(f"Final arm selection counts: Arm 1: {np.sum(choices==0)}, "
      f"Arm 2: {np.sum(choices==1)}, Arm 3: {np.sum(choices==2)}")

### Résultats

Puisque l'identité du meilleur bras alterne entre les trois bras pour un nombre égal d'essais, un bon algorithme de prise de décision devrait sélectionner chaque bras à peu près le même nombre de fois. D'après le décompte final des bras, ce n'est pas le cas : le bras 1 a été choisi de manière disproportionnée par rapport aux deux autres, probablement parce qu'il bénéficie de l'avantage d'avoir été le bras correct dès le premier bloc.

In [ ]:
# Separate cell for plotting without relaunching simulation

# ---------------- User parameters to play with ------------------
window = 20 # size of smoothing window for performance
snapshots = [700, 3000] # two snapshots to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplot_mosaic(
    [['perf', 'perf'],
     ['beta1', 'beta2']],
    figsize=(14, 8)
)
colors = ['steelblue', 'tomato', 'green']

# ----------------------------------------------------------------
# TOP FIGURE: PERFORMANCE + RASTER

correct = (choices == best_arm_schedule).astype(float)
performance = np.convolve(correct, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes['perf'].plot(performance, color='steelblue', lw=2)
axes['perf'].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes['perf'].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes['perf'].set_xlabel('Trial')
axes['perf'].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes['perf'].set_title('Vanilla Thompson sampling in a non-stationary environment\n'
                        'Red dotted lines = block changes (reward probabilities switch)')
axes['perf'].legend()
axes['perf'].set_ylim(-bottom_margin, 1.05)
axes['perf'].set_xlim(0, len(choices))

# separator line between curve area and ticks
axes['perf'].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes['perf'].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices == arm_idx)[0]
    axes['perf'].vlines(
            trial_indices,
            y_center - tick_height * 0.5,
            y_center + tick_height * 0.5,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes['perf'].text(
            0 - 80, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes['perf'].axvline(b * BLOCK_SIZE, color='red', linestyle=':', alpha=0.7)
    axes['perf'].text(b * BLOCK_SIZE + 2, 0.95, f'Block {b+1}', color='red', fontsize=8)

# ------------------------------------------------------------------
# BOTTOM FIGURES : TWO SNAPSHOTS OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)
colors = ['steelblue', 'tomato', 'green']

for ax_key, trial in zip(['beta1', 'beta2'], snapshots):
    for i in range(n_arms):
        a, b = alphas[trial-1, i], betas[trial-1, i]
        axes[ax_key].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={prob_schedule[trial-1,i]})')
        axes[ax_key].axvline(prob_schedule[trial-1,i], color=colors[i], linestyle='--', alpha=0.5)
    axes[ax_key].set_xlabel('Reward probability')
    axes[ax_key].set_ylabel('Density')
    axes[ax_key].set_title(f'Beta distributions after {trial} trials')
    axes[ax_key].legend()    

plt.tight_layout()
plt.show()

Le Thompson Sampling standard se débrouille bien dans le premier bloc : il identifie très rapidement le meilleur bras et le sélectionne presque tout le temps. Mais après le changement de bloc, les performances chutent et se rétablissent plus lentement qu'elles n'avaient augmenté au départ au début du bloc 1, témoignant d'une perte de flexibilité.
**La raison**: à la fin du premier bloc, le bras 1 a accumulé de nombreux succès. Sa distribution beta est étroite et centrée près de 0,6. Même après le passage au bloc 2 et la diminution des récompenses du bras 1, il faut de nombreux essais infructueux pour déplacer vers la gauche cette distribution très confiante. On voit ainsi comment, 200 essais après le début du bloc 2 (essai 700), le pic de la distribution du bras 1 est encore bien plus à droite que celui du bras 2 — qui est pourtant le meilleur — lequel reste en outre très large, rendant d'autant plus probable le fait d'en tirer des échantillons inférieurs à ceux du bras 1. Au bloc 3, les performances se rétablissent en réalité beaucoup plus vite qu'au bloc 2, mais moins vite qu'au bloc 1; cela tient à la combinaison de deux facteurs : la distribution du bras 1 a déjà eu 500 essais du bloc 2 pour se décaler vers la gauche, tandis que la distribution du bras 2 n'a pas pu s'étoffer autant que celle du bras 1 à la fin du bloc 1, en raison des faibles performances au cours du bloc 2. Le bloc 4 est un répétition du bloc 1, et malgré l'expérience déjà acquise lors de ce bloc, les performances du Thompson Sampling sont moins bonnes qu'à l'origine, car la concurrence avec les bras 2 et 3, dont les distributions sont désormais bien plus proches de 0,6, est devenue plus difficile. Des interprétations similaires s'appliquent aux blocs 5 et 6, qui répètent respectivement les blocs 2 et 3. À la fin de l'expérience (essai 3000), les trois distributions se chevauchent considérablement, rendant la sélection très aléatoire.

Cela motive deux extensions conçues spécifiquement pour les environnements non-stationnaires.

---
## Comment s'attaquer aux environnements non-stationnaires

### Sliding-Window Thompson Sampling (SWTS)

Proposé par Trovo et al. (2020), le SWTS (*Sliding-Window* signifie fenêtre glissante) adopte une approche très directe pour résoudre ce problème. Fondamentalement, le Thompson Sampling est peu performant parce qu'il se souvient trop bien : les anciens résultats comptent autant que les récents pour façonner les distributions beta (contrairement au Q-learning, qui pondère les résultats en fonction de leur ancienneté, un sujet que nous pourrions explorer une autre fois), si bien qu'il faut des quantités toujours plus grandes de preuves pour contrebalancer les preuves accumulées. Ce qu'il faut, c'est un mécanisme de filtrage garantissant que les résultats récents comptent davantage. Comme son nom l'indique, l'approche par fenêtre glissante prend une voie radicale en ne comptabilisant que les observations des T derniers essais. Les résultats dans cette fenêtre ont tous le même poids, mais au moins ils ne s'accumulent pas indéfiniment. La taille de la fenêtre glissante, T, contrôle la flexibilité de l'algorithme : si elle est petite, l'algorithme apprend vite mais est bruité; si elle est grande, il s'adapte plus lentement mais est plus stable.

In [ ]:
def sliding_window_thompson_sampling(n_trials, prob_schedule, T):
    """
    Sliding Window Thompson Sampling (Trovo et al., 2020).
    """
    
    # initialise beta distribution parameters
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    
    # trial-by-trial storage of results
    choices = np.zeros(n_trials, dtype = int)
    rewards = np.zeros(n_trials, dtype = int)
    alphas = np.zeros((n_trials, n_arms))
    betas = np.zeros((n_trials, n_arms))
    
    for t in range(n_trials):

        # step 1: Sample
        samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
        
        # step 2 : select arm that gave the largest sample
        choice = np.argmax(samples)

        # step 3 : pull arm and observe
        reward = pull_arm(choice, prob_schedule[t])
        
        # step 4: update the Beta distribution of the chosen arm
        alpha[choice] += reward
        beta[choice] += 1 - reward

        # step 5: forget the last trial outside of window size 
        if t >= T: # the sliding-window only starts operating after T trials
            edge = t - T # the last trial to slip out of the window
            past_choice = choices[edge]
            past_rwd = rewards[edge]

            if past_rwd == 1:
                alpha[past_choice] -= 1
            else:
                beta[past_choice] -= 1

        # store results of this trial
        choices[t] = choice
        rewards[t] = reward
        alphas[t] = alpha
        betas[t] = beta
    
    return choices, rewards, alphas, betas

# Run SWTS on the non-stationary task
T = 100
choices_SWTS, rewards_SWTS, alphas_SWTS, betas_SWTS = sliding_window_thompson_sampling(N_TRIALS_NS, prob_schedule, T)

print(f"Final arm selection counts: Arm 1: {np.sum(choices_SWTS==0)}, "
      f"Arm 2: {np.sum(choices_SWTS==1)}, Arm 3: {np.sum(choices_SWTS==2)}")

Cette fois-ci, le décompte final entre les trois bras est bien mieux équilibré.

In [ ]:
# Separate cell for plotting without relaunching simulation

# ---------------- User parameter to play with ------------------
window = 20 # size of smoothing window for performance
snapshots = [700, 3000] # two snapshots to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplot_mosaic(
    [['perf', 'perf'],
     ['beta1', 'beta2']],
    figsize=(14, 8)
)

# ----------------------------------------------------------------
# TOP FIGURE: PERFORMANCE + RASTER

correct = (choices_SWTS == best_arm_schedule).astype(float)
performance = np.convolve(correct, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes['perf'].plot(performance, color='steelblue', lw=2)
axes['perf'].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes['perf'].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes['perf'].set_xlabel('Trial')
axes['perf'].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes['perf'].set_title(f'Sliding-Window Thompson sampling in a non-stationary environment (T = {T})\n'
                        'Red dotted lines = block changes (reward probabilities switch)')
axes['perf'].legend()
axes['perf'].set_ylim(-bottom_margin, 1.05)
axes['perf'].set_xlim(0, len(choices_SWTS))

# horizontal separator line between curve area and ticks
axes['perf'].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes['perf'].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices_SWTS == arm_idx)[0]
    axes['perf'].vlines(
            trial_indices,
            y_center - tick_height*0.5,
            y_center + tick_height*0.5,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes['perf'].text(
            0 - 50, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes['perf'].axvline(b * BLOCK_SIZE, color='red', linestyle=':', alpha=0.7)
    axes['perf'].text(b * BLOCK_SIZE + 2, 0.95, f'Block {b+1}', color='red', fontsize=8)

# ------------------------------------------------------------------
# BOTTOM FIGURES : TWO SNAPSHOTS OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)
colors = ['steelblue', 'tomato', 'green']

for ax_key, trial in zip(['beta1', 'beta2'], snapshots):
    for i in range(n_arms):
        a, b = alphas_SWTS[trial-1, i], betas_SWTS[trial-1, i]
        axes[ax_key].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={prob_schedule[trial-1,i]})')
        axes[ax_key].axvline(prob_schedule[trial-1,i], color=colors[i], linestyle='--', alpha=0.5)
    axes[ax_key].set_xlabel('Reward probability')
    axes[ax_key].set_ylabel('Density')
    axes[ax_key].set_title(f'Beta distributions after {trial} trials')
    axes[ax_key].legend()    

plt.tight_layout()
plt.show()

Bien que les performances ne soient jamais aussi stables qu'elles l'étaient au bloc 1 pour le Thompson Sampling standard, le SWTS effectue correctement les transitions entre bras conformément au changement des probabilités. En effet, les frontières des blocs peuvent se deviner dans le tracé raster, qui dépeint clairement des périodes alternées de domination de chaque bras. Le bruit dans la sélection des bras est dû aux distributions beta sous-jacentes, qui restent très larges même en fin de bloc. Par exemple, après 3000 essais, l'algorithme vient de terminer 500 essais d'un bloc où le bras 3 était le meilleur levier ; et pourtant, même si la distribution du bras 3 est effectivement très proche de 0,6, les deux bras concurrents ont des distributions si larges que la sélection de ces options sous-optimales se produit encore relativement souvent.

### Dynamic Thompson Sampling (DTS)

Quoique plus ancien que le SWTS, le Dynamic Thompson Sampling, introduit dans Gupta et al. (2011), adopte une approche plus sophistiquée. Au lieu d'écarter les essais anciens, le DTS se concentre sur les distributions beta elles-mêmes, en les contraignant à conserver une largeur minimale. Pour mettre cela en œuvre, rappelons comment la largeur des distributions bêta dépend de $\alpha + \beta$: plus cette somme est grande, plus le pic est étroit autour de la moyenne. Ainsi, pour garantir une largeur minimale — ou variance en termes techniques — à chaque essai, après avoir observé le résultat de la sélection du bras i, le DTS vérifie que $\alpha_i + \beta_i < C$, avec C un paramètre prédéfini. Si la vérification est satisfaite, $\alpha_i$ et $\beta_i$ sont mis à jour comme d'habitude. Dans le cas contraire, la règle de mise à jour devient:
   - Si récompense = 1: $\alpha_i \leftarrow (\alpha_i + 1) * C / (C+1)$; $\beta_i \leftarrow \beta_i * C / (C+1)$
   - Si récompense = 0: $\beta_i \leftarrow (\beta_i + 1) * C/(C+1)$; $\alpha_i \leftarrow \alpha_i * C / (C+1)$

Vous pouvez facilement vérifier par vous-même que cette modification garantit que $\alpha_i + \beta_i <= C$. Il est intéressant de noter comment, contrairement au Thompson Sampling standard, les deux paramètres de forme sont mis à jour simultanément selon cette règle. Autre différence notable par rapport au SWTS: au lieu d'une contrainte globale sur le nombre total d'observations passées, la vérification est effectuée séparément pour chaque bras ; ainsi, ne pas sélectionner un bras pendant longtemps n'entraîne pas une dégradation de sa distribution vers l'uniforme, ce qui le rendrait plus susceptible d'être testé.
Sur le plan mathématique, cette mise à jour garantit que la variance est contrainte. Pour les plus mathématiquement enclins, il est sans doute plus naturel de raisonner en termes de variance minimale souhaitée, puis de remonter à la valeur de C. Une autre propriété mathématique intéressante de cette mise à jour, que vous pouvez voir démontrée dans l'article original de Gupta et al. (2011), est qu'elle attribue implicitement des poids exponentiels décroissants aux résultats passés, de sorte que les plus récents sont pondérés plus fortement que les anciens.

In [ ]:
def dynamic_thompson_sampling(n_trials, prob_schedule, C):
    """
    Dynamic Thompson Sampling (Gupta et al., 2011).
    """
    
    # initialise beta distribution parameters
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    
    # trial-by-trial storage of results
    choices = np.zeros(n_trials, dtype = int)
    rewards = np.zeros(n_trials, dtype = int)
    alphas = np.zeros((n_trials, n_arms))
    betas = np.zeros((n_trials, n_arms))
    
    for t in range(n_trials):

        # step 1: Sample
        samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
        
        # step 2 : select arm that gave the largest sample
        choice = np.argmax(samples)

        # step 3 : pull arm and observe
        reward = pull_arm(choice, prob_schedule[t])
        
        # step 4: update the Beta distribution of the chosen arm
        if alpha[choice] + beta[choice] < C:
            alpha[choice] += reward 
            beta[choice] += 1 - reward
        else:
            alpha[choice] = (alpha[choice] + reward) * C / (C+1) 
            beta[choice] = (beta[choice] + 1 - reward) * C / (C+1)

        # store results of this trial
        choices[t] = choice
        rewards[t] = reward
        alphas[t] = alpha
        betas[t] = beta
    
    return choices, rewards, alphas, betas

# Run DTS on the non-stationary task
C = 20
choices_DTS, rewards_DTS, alphas_DTS, betas_DTS = dynamic_thompson_sampling(N_TRIALS_NS, prob_schedule, C)

print(f"Final arm selection counts: Arm 1: {np.sum(choices_DTS==0)}, "
      f"Arm 2: {np.sum(choices_DTS==1)}, Arm 3: {np.sum(choices_DTS==2)}")

De même que pour le SWTS, chaque bras a été sélectionné grosso modo le même nombre de fois, indiquant une bonne flexibilité de l'algorithme.

In [ ]:
# Separate cell for plotting without relaunching simulation

# ---------------- User parameter to play with ------------------
window = 20 # size of smoothing window for performance
snapshots = [700, 3000] # two snapshots to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplot_mosaic(
    [['perf', 'perf'],
     ['beta1', 'beta2']],
    figsize=(14, 8)
)

# ----------------------------------------------------------------
# TOP FIGURE: PERFORMANCE + RASTER

correct = (choices_DTS == best_arm_schedule).astype(float)
performance = np.convolve(correct, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes['perf'].plot(performance, color='steelblue', lw=2)
axes['perf'].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes['perf'].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes['perf'].set_xlabel('Trial')
axes['perf'].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes['perf'].set_title(f'Dynamic Thompson sampling in a non-stationary environment (C = {C})\n'
                        'Red dotted lines = block changes (reward probabilities switch)')
axes['perf'].legend()
axes['perf'].set_ylim(-bottom_margin, 1.05)
axes['perf'].set_xlim(0, len(choices_DTS))

# horizontal separator line between curve area and ticks
axes['perf'].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes['perf'].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices_DTS == arm_idx)[0]
    axes['perf'].vlines(
            trial_indices,
            y_center - tick_height * 0.5,
            y_center + tick_height * 0.5,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes['perf'].text(
            0 - 50, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes['perf'].axvline(b * BLOCK_SIZE, color='red', linestyle=':', alpha=0.7)
    axes['perf'].text(b * BLOCK_SIZE + 2, 0.95, f'Block {b+1}', color='red', fontsize=8)

# ------------------------------------------------------------------
# BOTTOM FIGURES : TWO SNAPSHOTS OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)
colors = ['steelblue', 'tomato', 'green']

for ax_key, trial in zip(['beta1', 'beta2'], snapshots):
    for i in range(n_arms):
        a, b = alphas_DTS[trial-1, i], betas_DTS[trial-1, i]
        axes[ax_key].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={prob_schedule[trial-1,i]})')
        axes[ax_key].axvline(prob_schedule[trial-1,i], color=colors[i], linestyle='--', alpha=0.5)
    axes[ax_key].set_xlabel('Reward probability')
    axes[ax_key].set_ylabel('Density')
    axes[ax_key].set_title(f'Beta distributions after {trial} trials')
    axes[ax_key].legend()    

plt.tight_layout()
plt.show()

La sélection du bras 1 dans le bloc 1 n'est peut-être pas aussi exclusive qu'avec le Thompson Sampling classique, mais elle est très élevée, et bien plus stable que celle du SWTS. Comme pour le SWTS, les transitions entre blocs sont gérées de manière fluide, avec une chute et un rétablissement très rapides de la performance. Les distributions beta sous-jacentes à l'essai 3000 montrent leur grande largeur tout en restant clairement séparées — contrairement au SWTS — ce qui signifie qu'elles sont à la fois suffisamment discriminantes pour éviter une exploration superflue, et facilement actualisables en cas de changement de bloc. Cela est en partie dû au fait que les probabilités de récompense ont été choisies suffisamment différentes: si elles avaient été plus proches, par exemple des probabilités de récompense de 0,4 contre 0,6, une valeur plus élevée de C aurait pu s'avérer nécessaire. Cela nous amène à une réflexion plus générale sur le choix des paramètres. Dans ces exemples, T et C ont été choisis manuellement, de sorte que comparer les deux algorithmes comporte une réserve importante : ces différences sont-elles systématiques ou peuvent-elles être atténuées par des choix différents ? De plus, ces choix influencent également les performances dans l'environnement de test. Garantir de bonnes performances dans un environnement plus bruité avec des rotations de blocs plus fréquentes nécessiterait un réglage fin des paramètres — un problème qui n'existe pas pour le Thompson Sampling classique.

---

## Références

- N. Gupta, O. -C. Granmo and A. Agrawala (2011) Thompson Sampling for Dynamic Multi-armed Bandits. 10th International Conference on Machine Learning and Applications and Workshops, 2011, pp. 484-489. Disponible à https://www.researchgate.net/publication/232616670_Thompson_Sampling_for_Dynamic_Multi-armed_Bandits
- Trovo, F., Paladino, S., Restelli, M., & Gatti, N. (2020). Sliding-window Thompson sampling for non-stationary settings.
Journal of Artificial Intelligence Research, 68, 311–364. Available at:. https://doi.org/10.1613/jair.1.11407
- Cinotti, F., Coutureau, E., Khamassi, M., Marchand, A. R., & Girard, B. (2024). Regulation of reinforcement learning parameters captures long-term changes in rat behaviour. European Journal of Neuroscience, 60(4), 4469–4490. https://doi.org/10.1111/ejn.16449

